# Лекция: Распознавание изображений

Для этого раздела понадобятся следующие модули Python. Убедитесь, что они установлены.
- `matplotlib`
- `requests`
- `numpy`
- `io`
- `pathlib`
- `tensorflow`


## Необходимая инициализация и вспомогательные функции

Перед началом необходимо выполнить некоторую инициализацию.


In [ ]:
import tensorflow as tf

# This initialization code is required due to an error
# "NotFoundError: No algorithm worked"
# when using Conv2D
# Probabliy due to problems with cuda 11.
# Remove this when fixed
# https://github.com/tensorflow/tensorflow/issues/43174

# from tensorflow.compat.v1 import ConfigProto
# from tensorflow.compat.v1 import InteractiveSession

# config = ConfigProto()
# config.gpu_options.allow_growth = True
# session = InteractiveSession(config=config)

Функция `plot_hist`, приведённая ниже, строит кривые обучения: зависимость функции потерь и точности от номера эпохи.

Она принимает список историй обучения `hist_list`, вычисленных для разных сетей, и отображает их в виде ряда графиков.


In [ ]:
import matplotlib.pyplot as plt

def plot_hist_one(hist, name, axs):
    """Plot one loss and accuracy"""
    epochs = len(hist.history['loss'])
    xs = list(range(epochs))

    ax = axs[0]
    ax.plot(xs, hist.history['loss'], label='loss')
    ax.plot(xs, hist.history['val_loss'], label='val_loss')
    ax.set_ylabel('loss')
    ax.set_yscale('log')

    ax = axs[1]
    ax.plot(xs, hist.history['accuracy'], label='accuracy');
    ax.plot(xs, hist.history['val_accuracy'], label='val_accuracy');
    ax.set_ylabel('accuracy')

    for ax in axs:
        ax.grid()
        ax.set_xlabel('epoch')
        ax.legend(title=name)

def plot_hist(hist_list, hist_names):
    """Plot loss and accuracy for many network"""
    N = len(hist_list)
    fig, axs = plt.subplots(nrows=N, ncols=2, figsize=(10, 3*N))
    if N == 1:
        axs = [axs]
    for hist, name, ax in zip(hist_list, hist_names, axs):
        plot_hist_one(hist, name, ax)
    plt.tight_layout()

## Изображение с точки зрения компьютера

Изображение в градациях серого — это двумерный массив.

Каждая ячейка такого массива соответствует пикселю изображения.

Значения кодируют уровень яркости.

Обычно каждый пиксель кодируется одним байтом.

Напомним, что байт содержит 8 бит и, следовательно, может хранить целые числа от 0 до 255.

В NumPy байты представлены типом `np.uint8` - беззнаковое 8-ми битное целое.

Таким образом, изображение в градациях серого — это двумерный массив целых чисел типа `np.uint8`. Каждое число лежит в диапазоне от 0 до 255: 0 соответствует чёрному цвету, а 255 — белому.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

im1 = np.zeros((3, 3), dtype=np.uint8)

im1[1, 1] = 255
im1[0, 1] = 255
im1[1, 0] = 255
im1[2, 1] = 255
im1[1, 2] = 255

print(im1)

Изображение можно отобразить с помощью функции `imshow` из библиотеки `matplotlib`:


In [ ]:
fig, ax = plt.subplots()
ax.imshow(im1, cmap='gray');

Когда изображение планируется обрабатывать нейронной сетью, его обычно преобразуют к типу с плавающей точкой, в котором значения лежат в диапазоне от 0 до 1.

`Matplotlib` всё равно может корректно отобразить такое масштабированное изображение, поскольку автоматически определяет масштаб.


In [ ]:
im2 = im1 / 256.0
print(im2)

fig, ax = plt.subplots()
ax.imshow(im2, cmap='gray');

Пиксели изображения нумеруются начиная с верхнего левого угла. Сначала идёт вертикальная координата, то есть высота, а затем горизонтальная координата, то есть ширина.

В примере ниже мы создаём изображение высотой 17 пикселей и шириной 5 пикселей.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

im3 = np.zeros((17, 5), dtype=np.uint8)

cnt = 0
for y in range(17):
    for x in range(5):
        if y % 5 == x:
            cnt += 1
            im3[y, x] = cnt

print(im3)

fig, ax = plt.subplots(figsize = (2, 16))
ax.imshow(im3, cmap='gray');

Если хранить изображение как массив NumPy, его размеры можно получить через атрибут `.shape`.


In [ ]:
print(im3.shape)

Цвета на экране компьютера формируются как смесь нескольких каналов.

Одна из популярных схем кодирования — RGB: каждый пиксель кодируется тремя байтами, то есть тремя целыми числами, каждое из которых лежит в диапазоне от 0 до 255.

Цветное изображение в модели RGB можно рассматривать как стопку из трёх слоёв в градациях серого. Каждый слой кодирует интенсивность красного, зелёного и синего цветов.

В итоге цветное изображение представляется в памяти компьютера как трёхмерный массив.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

im4 = np.zeros((32, 32, 3), dtype=np.uint8)
size_y, size_x, _ = im4.shape

# Red channel - intensity grows vertically
cnt = 0
for y in range(size_y):
    for x in range(size_x):
        im4[y, x, 0] = cnt
        cnt = (cnt + 1) % 256

# Green channel - intensity grows horizontally
cnt = 0
for x in range(size_x):
    for y in range(size_y):
        im4[y, x, 1] = cnt
        cnt = (cnt + 1) % 256

# Blue channel - along diagonals
cnt = 0
for s in range(0, size_x + size_y - 1):
    for x in range(max(0, s - (size_y - 1)), min(s + 1, size_x)):
        y = s - x
        im4[y, x, 2] = cnt
        cnt = (cnt + 1) % 256

print("Image sizes: ", im4.shape)

fig, axs = plt.subplots(nrows=1, ncols=4, figsize=(20, 5))
axs[0].imshow(im4[:,:,0], cmap='gray')
axs[1].imshow(im4[:,:,1], cmap='gray')
axs[2].imshow(im4[:,:,2], cmap='gray')
axs[3].imshow(im4);

Можно также учитывать прозрачность. Тогда, помимо перечисленных, появляется ещё один канал, который называется альфа-каналом.

Ноль означает полную прозрачность, а 255 — отсутствие прозрачности.

Существует и много других способов кодирования цветов. Они называются цветовыми моделями: HSV, LAB и др.


## Бинарная кросс-энтропия

Ранее мы рассматривали бинарную классификацию. Сеть имела один выходной нейрон со ступенчатой функцией или сигмоидой в качестве функции активации.

Результатом было значение $0\leq q\leq 1$, которое сравнивалось с истинным значением $p$, равным либо точно $0$, либо $1$.

Функцией потерь была среднеквадратичная ошибка:

$$
\text{MSE} = \frac{1}{N}\sum_{i=1}^N (p_i - q_i)^2
$$

Хотя это кажется интуитивно очевидным выбором, среднеквадратичная ошибка не является наилучшей функцией потерь для задач классификации.

Обычно вместо неё используют кросс-энтропию.

Напомним её смысл.

Мы по-прежнему обсуждаем бинарное предсказание, то есть случай двух классов: 0 и 1.

Выход сети $q$ можно интерпретировать как вероятность того, что некоторый объект данных принадлежит классу 1.

Следовательно, у нас есть распределение вероятностей: объект принадлежит классу 1 с вероятностью $q$ и классу 0 — с вероятностью $(1-q)$.

Кроме того, у нас есть истинное распределение вероятностей: класс 1 имеет вероятность $p$, а класс 0 — вероятность $(1-p)$. Не будем забывать, что $p$ на самом деле может быть только 1 или 0.

Кросс-энтропия — это функция, сравнивающая два распределения вероятностей и показывающая, насколько они различаются.

В бинарном случае она называется бинарной кросс-энтропией и имеет вид:

$$
H_p(q) = -p\log q - (1-p)\log(1-q)
$$

Рассмотрим объект данных, истинный класс которого равен $p=1$, а сеть предсказала для него вероятность $q$.

Функция потерь должна быть равна нулю, если $q=1$, и должна возрастать при $q<1$.

В этом случае процедура обучения будет стремиться сделать $q$ ближе к 1.

Если же сеть предсказывает значение $q$, очень близкое к 0, то есть работает совершенно неверно, штраф должен быть действительно большим.

Сравним штрафы за ошибочные предсказания, которые дают MSE и бинарная кросс-энтропия.

Для MSE при $p=1$ отклонение от истинного значения равно

$$
P_\text{MSE}=(1-q)^2
$$

А для кросс-энтропии имеем:

$$
P_\text{BCE}=-\log q
$$

Построим графики и сравним штрафы:


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

q = np.linspace(1e-5, 1-1e-5, 100)
penalty_mse = (1-q)**2
penalty_log = -np.log(q)

fig, ax = plt.subplots()
ax.plot(q, penalty_mse, label="MSE")
ax.plot(q, penalty_log, label="Binary cross entropy")
ax.grid()
ax.legend();

Мы ясно видим, что бинарная кросс-энтропия задаёт более подходящие штрафы за неверные предсказания.

Именно поэтому кросс-энтропия обычно используется как функция потерь при обучении сети в качестве классификатора.


## Простой пример: рукописные цифры 0 и 1

Мы создадим очень простую нейронную сеть, способную распознавать рукописные цифры 0 и 1.

Для обучения сети будем использовать часть набора данных MNIST.

Набор данных MNIST (Modified National Institute of Standards and Technology) представляет собой большой набор изображений рукописных цифр, который широко используется для обучения систем обработки изображений.

MNIST содержит 60000 обучающих изображений и 10000 тестовых изображений. Каждое изображение — это изображение в градациях серого размером 28 × 28 пикселей.

Фактически MNIST является стандартным набором данных для изучения методов машинного обучения.

TensorFlow предоставляет функцию `.mnist.load_data()` для загрузки и открытия копии этого набора данных.

При первом вызове функции набор данных загружается в локальный каталог по умолчанию.

В Linux и macOS это `$HOME/.keras/`, а в Windows — `%USERPROFILE%\.keras\`.

При последующих вызовах функция просто открывает уже загруженный набор данных.


In [ ]:
# Returns tuple of numpy arrays: (x_train, y_train), (x_test, y_test)
(X_train, y_train), (X_test, y_test) = tf.keras.datasets.mnist.load_data()
print("Image labels example: ", y_train[:20])

На данный момент мы будем рассматривать только два класса, поэтому извлечём изображения, помеченные как `0` или `1`.


In [ ]:
import numpy as np

def extract01(X_data, y_data):
    """Extract images with 0 or 1 only"""
    X_data01 = []
    y_data01 = []
    for X, y in zip(X_data, y_data):
        if y <= 1:
            X_data01.append(X)
            y_data01.append(y)
    return np.array(X_data01), np.array(y_data01)

X_train01, y_train01 = extract01(X_train, y_train)
X_test01, y_test01 = extract01(X_test, y_test)

print("train ", X_train01.shape)
print("test ", X_test01.shape)

Посмотрим, что у нас получилось: ниже показано несколько изображений из набора данных.


In [ ]:
import matplotlib.pyplot as plt

fig, axs = plt.subplots(nrows=10, ncols=16)

for ax, X, y in zip(axs.reshape(-1), X_train01, y_train01):
    ax.imshow(X, cmap='gray_r')
    ax.axis("off")

plt.tight_layout()

Проверим баланс классов.

Рассматривая гистограммы меток, мы видим, что все классы встречаются почти с одинаковой частотой.


In [ ]:
import matplotlib.pyplot as plt

fig, axs = plt.subplots(nrows=1, ncols=2, figsize=(10, 4))

axs[0].hist(y_train01, bins=2, rwidth=0.8)
axs[1].hist(y_test01, bins=2, rwidth=0.8);

Теперь можно создать модель. Заметим, что на вход сети мы будем подавать пакет изображений в градациях серого. Это трёхмерный массив `[batch_size, image_height, image_width]` из беззнаковых 8‑битных целых чисел.


In [ ]:
print("Batch shape: ", X_train01.shape, "\nBatch type: ", X_train01.dtype)

Однако полносвязные (dense) слои, которые мы будем использовать, принимают двумерные пакеты.

Поэтому необходимо преобразовать двумерные изображения в плоские одномерные массивы: `[batch_size, image_height * image_width]`.

Это делается с помощью слоя `Flatten`.

Также напомним, что большинство моделей машинного обучения работают лучше, когда на вход подаются стандартизированные данные.

Для этого добавляется слой `Lambda`, в котором выполняется масштабирование данных.

Сама сеть максимально проста. Она имеет два dense‑слоя: скрытый и выходной. В скрытом слое всего 2 нейрона.

Выходной слой имеет один нейрон, поскольку выполняется бинарная классификация.

Его функция активации не задаётся, а функция потерь имеет параметр `from_logits=True`. Это означает, что последняя активация вычисляется одновременно с функцией потерь.

Такая организация вычислений позволяет уменьшить численные ошибки.

Здесь и далее используется оптимизатор `Adam`.


In [ ]:
input_shape = X_train01.shape[1:]

model_mnist01 = tf.keras.models.Sequential([
    tf.keras.layers.Input(input_shape),
    tf.keras.layers.Lambda(lambda data: data / 256.0),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(2, activation='sigmoid'),
    tf.keras.layers.Dense(1)
])

model_mnist01.compile(optimizer=tf.keras.optimizers.Adam(),
                      loss=tf.keras.losses.BinaryCrossentropy(from_logits=True),
                      metrics=['accuracy'])

model_mnist01.summary()

Обратите внимание, что мы задаём форму входных данных для первого слоя сети. Это форма одной записи данных, то есть `(28, 28)`, а не всего пакета.

Это не обязательно, но позволяет вывести сводку модели без её предварительного запуска.

Если форма входа не указана, модель не знает, сколько у неё входов, и нейроны фактически создаются только при первом вычислении модели.

Если мы хотим увидеть сводку модели до её использования, нужно указать форму входного тензора.


Запустим обучение.


In [ ]:
hist_mnist01 = model_mnist01.fit(X_train01, y_train01, epochs=50, validation_split=0.2, verbose=1)

Вот кривые обучения.

Мы видим, что модель обучается очень быстро и достигает почти 100 % точности.


In [ ]:
plot_hist([hist_mnist01], ["mnist01"])

Это оценка на тестовых данных. Результат получается довольно хорошим.


In [ ]:
loss, acc = model_mnist01.evaluate(X_test01, y_test01)
print(f"acc={acc}, loss={loss}")

Тем не менее можно заметить переобучение: функция потерь для обучающих данных продолжает уменьшаться в правой части графиков, тогда как функция потерь для валидационных данных выходит на плато.

В данном случае это не является проблемой, поскольку переобучение проявляется уже после достижения очень высокой точности даже на валидационных и тестовых данных.

Наконец посмотрим визуально на качество предсказаний.

Здесь показаны предсказания для тестовых данных. Обратите внимание, что результат является тензором чисел с плавающей точкой. Отрицательные значения соответствуют классу 0, а положительные — классу 1.

Чтобы использовать предсказания, мы преобразуем тензор в массив NumPy.


In [ ]:
y_pred = model_mnist01(X_test01, training=False)
print("y_pred tensor=", y_pred, "\n\ny_pred numpy=", y_pred.numpy())

Теперь построим несколько изображений вместе с предсказанными значениями. Мы видим, что предсказания получаются очень точными.


In [ ]:
fig, axs = plt.subplots(nrows=8, ncols=8, figsize=(10, 10))

for ax, X, y in zip(axs.ravel(), X_test01, y_pred.numpy()):
    ax.imshow(X, cmap='gray_r')
    ax.set_title(f"{0 if y[0] < 0 else 1}")
    ax.axis("off")

plt.tight_layout()

Почти идеальный результат при такой простой сети объясняется тем, что сам набор данных очень простой.

Оказывается, различать рукописные нули и единицы довольно легко.


## Функции argmax и softmax для многоклассовой классификации

Если сеть выполняет бинарную классификацию, она обычно имеет один выходной нейрон с сигмоидальной функцией активации, а в качестве функции потерь используется бинарная кросс‑энтропия.

Для многоклассовой классификации число выходных нейронов должно быть равно числу классов.

Каждый нейрон соответствует определённому классу.

Предсказанный класс — это тот, чей нейрон выдаёт наибольшее значение.

Таким образом, имея логиты (напомним, что логиты — это сырые выходы нейронов без функции активации), необходимо найти индекс наибольшего из них.

Если имеется $N$ классов и $y_i$, $i\in[0, N-1]$, — логиты сети, то номер предсказанного класса $i_{pred}$ можно записать так:

$$
i_{pred} = \operatorname{argmax}_{i\in[0,N-1]} y_i
$$

Эта запись означает, что мы рассматриваем все $y_i$, находим наибольшее значение и возвращаем его индекс $i$.

Однако эта процедура используется только после обучения сети.

Её нельзя использовать как функцию активации внутри функции потерь, поскольку она не является дифференцируемой.

Поэтому используется её гладкий аналог — функция softmax.

Softmax — это векторная функция, число элементов которой равно числу классов.

Её элемент для класса $k$ имеет вид:

$$
q_k = \frac{e^{y_k}}{\sum_{i=0}^{N-1} e^{y_i}}
$$

Можно доказать, что $q_k$ будет наибольшим тогда и только тогда, когда соответствующее значение $y_k$ является наибольшим.

Набор значений $q_k$, $k=0,1,\ldots,N-1$, можно рассматривать как распределение вероятностей, поскольку $q_k\in[0,1]$ и

$$
\sum_{k=0}^{N-1} q_k = 1
$$

Таким образом, сеть вычисляет вероятности принадлежности объекта каждому классу. Предсказанным считается класс с наибольшей вероятностью.

При обучении эти вероятности сравниваются с истинным распределением $p_k$, где для всех классов стоят нули, а для одного — единица (это называется one‑hot представлением меток классов).

Сравнение выполняется с помощью кросс‑энтропии, которая используется как функция потерь:

$$
H_p(q) = -\sum_{i=0}^{N-1} p_i \log q_i
$$

Таким образом, типичные правила для многоклассовой классификации (где $N$ — число классов):

- сеть имеет $N$ выходных нейронов  
- выходной слой используется без функции активации  
- функция потерь — кросс‑энтропия с логитами (softmax вычисляется одновременно с кросс‑энтропией для лучшей численной устойчивости)  
- при предсказании применяется функция argmax к выходному вектору сети


## Функция активации ReLU

Существует множество других функций, которые можно использовать в качестве активаций в глубоких сетях.

Иногда более высокая производительность достигается, если вместо сигмоиды в скрытых слоях используется функция Rectified Linear Unit (ReLU):

$$
\mathrm{ReLU}(y) = \max(0, y)
$$

Эта функция равна нулю при $y \le 0$ и возвращает значение $y$, когда $y > 0$.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

xs = tf.constant(np.linspace(-5, 5, 100), dtype = tf.float32)
ys1 = tf.keras.activations.sigmoid(xs)
ys2 = tf.keras.activations.relu(xs)

fig, ax = plt.subplots()

ax.plot(xs.numpy(), ys1.numpy(), label='sigmoid')
ax.plot(xs.numpy(), ys2.numpy(), label='relu')
ax.set_ylim([0, 3])
ax.legend()
ax.grid()

## Рукописные цифры от 0 до 9

Рассмотрим задачу распознавания всех цифр из набора данных MNIST.


In [ ]:
# Returns tuple of numpy arrays: (x_train, y_train), (x_test, y_test)
(X_train, y_train), (X_test, y_test) = tf.keras.datasets.mnist.load_data()
print("Image labels example: ", y_train[:20])

Согласно обсуждению выше, наша сеть будет вычислять вероятности того, что изображение принадлежит каждому из классов.

Следовательно, истинные метки классов также должны быть преобразованы в распределения вероятностей, где правильный класс имеет вероятность 1, а все остальные — вероятность 0.

Это называется one-hot представлением.


In [ ]:
import numpy as np

def onehot(y_labels, size):
    y_onehot = np.zeros((y_labels.shape[0], size))
    for ylb, yoh in zip(y_labels, y_onehot):
        yoh[ylb] = 1
    return y_onehot

# This is just an example how it works
y_onehot = onehot(y_train[:15], 10)
for ylb, yoh in zip(y_train[:15], y_onehot):
    print(ylb, yoh)

На самом деле такая функция уже существует в `tensorflow`. Она называется `.to_categorical`. Ниже мы будем её использовать.

Эта функция может автоматически определить число классов, поэтому её можно использовать всего с одним параметром.


In [ ]:
from tensorflow.keras.utils import to_categorical as to_categ

y_onehot = to_categ(y_train[:15], 10)
for ylb, yoh in zip(y_train[:15], y_onehot):
    print(ylb, yoh)

Посмотрим, что у нас есть: ниже показано несколько изображений из набора данных.


In [ ]:
import matplotlib.pyplot as plt

fig, axs = plt.subplots(nrows=10, ncols=16)

for ax, X, y in zip(axs.reshape(-1), X_train, y_train):
    ax.imshow(X, cmap='gray_r')
    ax.axis("off")

plt.tight_layout()

Рассматривая гистограммы меток, мы видим, что набор данных хорошо сбалансирован: все классы встречаются почти с одинаковой частотой.


In [ ]:
import matplotlib.pyplot as plt

fig, axs = plt.subplots(nrows=1, ncols=2, figsize=(10, 4))

axs[0].hist(y_train, bins=10, rwidth=0.8)
axs[1].hist(y_test, bins=10, rwidth=0.8);

При построении модели мы масштабируем данные так, чтобы они находились в диапазоне $[0,1]$, и преобразуем двумерные изображения в одномерные массивы.

Выходной слой содержит 10 нейронов.

Функция активации на выходе не задаётся, а функция потерь имеет параметр `from_logits=True`. Это означает, что последняя активация (softmax) будет вычисляться вместе с функцией потерь.

Функция потерь называется `CategoricalCrossentropy` и соответствует кросс-энтропии, рассмотренной выше.

Мы рассмотрим две модели. Модель A — самая простая:


In [ ]:
input_shape = X_train.shape[1:]

model_mnist_A = tf.keras.models.Sequential([
    tf.keras.layers.Input(input_shape),
    tf.keras.layers.Lambda(lambda data: data / 256.0),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(10, activation='sigmoid'),
    tf.keras.layers.Dense(10)
])

model_mnist_A.compile(optimizer=tf.keras.optimizers.Adam(),
                      loss=tf.keras.losses.CategoricalCrossentropy(from_logits=True),
                      metrics=['accuracy'])

model_mnist_A.summary()

Модель B более сложная. Это глубокая модель, содержащая 5 полносвязных слоёв.

Скрытые слои используют функцию активации ReLU вместо сигмоиды.

Эксперименты показывают, что эта функция активации работает лучше.


In [ ]:
input_shape = X_train.shape[1:]

model_mnist_B = tf.keras.models.Sequential([
    tf.keras.layers.Input(input_shape),
    tf.keras.layers.Lambda(lambda data: data / 256.0),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(10, activation='relu'),
    tf.keras.layers.Dense(30, activation='relu'),
    tf.keras.layers.Dense(50, activation='relu'),
    tf.keras.layers.Dense(30, activation='relu'),
    tf.keras.layers.Dense(10)
])

model_mnist_B.compile(optimizer=tf.keras.optimizers.Adam(),
                      loss=tf.keras.losses.CategoricalCrossentropy(from_logits=True),
                      metrics=['accuracy'])

model_mnist_B.summary()

In [ ]:
hist_mnist_A = model_mnist_A.fit(X_train, to_categ(y_train), epochs=50, validation_split=0.2, verbose=1)

In [ ]:
hist_mnist_B = model_mnist_B.fit(X_train, to_categ(y_train), epochs=50, validation_split=0.2, verbose=1)

In [ ]:
plot_hist([hist_mnist_A, hist_mnist_B], ["mnist A", "mnist B"])

Это оценка качества на тестовых данных.


In [ ]:
loss, acc = model_mnist_A.evaluate(X_test, to_categ(y_test))
print(f"acc={acc}, loss={loss}")

In [ ]:
loss, acc = model_mnist_B.evaluate(X_test, to_categ(y_test))
print(f"acc={acc}, loss={loss}")

Результат может показаться хорошим. Но на самом деле производительность умеренная.

Распознавание MNIST — это своего рода соревнование. Лидеры демонстрируют точность выше 99 %: https://en.wikipedia.org/wiki/MNIST_database

Разумеется, мы могли бы ещё улучшить результат, если бы продолжили обучение. Мы провели только 50 эпох, что очень мало.


Важный момент, который нас интересует, — это переобучение.

Модель A имеет меньше нейронов, а значит её информационная ёмкость ниже.

Мы наблюдаем, что у неё меньше переобучение, но и ниже точность.

Если увеличить число нейронов (модель B), мы получаем более высокую точность, но переобучение становится более выраженным.

Ниже мы обсудим стандартный метод борьбы с переобучением в глубоких сетях.

Но прежде посмотрим, как работает предсказание.

Напомним: наша сеть возвращает логиты, то есть числа с плавающей точкой.

Чтобы получить предсказания, необходимо применить функцию $\text{argmax}$ к выходам сети.

Предсказанный класс соответствует наибольшему значению.


In [ ]:
y_logits = model_mnist_B(X_test, training=False)
print(y_logits[:3])

In [ ]:
y_pred = tf.math.argmax(y_logits, axis=1).numpy()
print(y_pred[:3])

In [ ]:
fig, axs = plt.subplots(nrows=8, ncols=8, figsize=(10, 10))

for ax, X, y in zip(axs.ravel(), X_test, y_pred):
    ax.imshow(X, cmap='gray_r')
    ax.set_title(f"{y}")
    ax.axis("off")

plt.tight_layout()

## Борьба с переобучением с помощью dropout

Во время обучения различные части сети могут слишком сильно подстроиться друг под друга.

Это приводит к переобучению, когда модель очень хорошо запоминает обучающие данные, но плохо обобщает новые.

Идея dropout заключается в том, чтобы перед каждым шагом обучения случайным образом изменять структуру сети.

Это делается путём случайного отключения нейронов с вероятностью $p$.

На рисунке ниже показаны полносвязные слои сети до применения dropout.


![dropout_before.png](https://raw.githubusercontent.com/kupav/data-sc-intro/refs/heads/main/fig/dropout_before.png)

На следующем рисунке показана одна из возможных конфигураций после случайного отключения некоторых нейронов.


![dropout_after.png](https://raw.githubusercontent.com/kupav/data-sc-intro/refs/heads/main/fig/dropout_after.png)

Предположим, что у нас есть нейрон $k$, который принимает вектор входов $x$, веса $W$ и смещение $b$:

$$
y_k = \sigma(x W + b)
$$

Этот нейрон является частью слоя из $K$ нейронов.

При использовании dropout каждый выход нейрона умножается на случайное число $d_k$:

$$
y_k = d_k \sigma(x W + b)
$$

Здесь $d_k$ принимает значение 1 с вероятностью $p$ и 0 с вероятностью $(1-p)$.

Эти случайные числа генерируются перед каждым шагом обучения, который включает вычисление градиентов и обновление весов.

Веса и смещения нейронов, отключённых на текущем шаге, не обновляются.

На следующем шаге обучения генерируются новые случайные значения $d_k$, поэтому отключаются уже другие нейроны.

Обучение с использованием dropout можно рассматривать как обучение ансамбля сетей.

Когда сеть используется для предсказаний, dropout отключается, и мы фактически получаем усреднённый результат этого ансамбля.


Теперь посмотрим, как работает dropout. В `tensorflow` с использованием `keras` достаточно добавить слой `Dropout`.

Его параметр `rate` задаёт вероятность отключения нейронов.

Существуют разные способы добавления dropout-слоёв. Можно добавить один такой слой с большим значением вероятности, а можно добавлять dropout после каждого слоя нейронов. В этом случае вероятность отключения должна быть меньше.

Ниже мы возьмём модель B и добавим dropout-слои после каждого dense-слоя с небольшим значением параметра.


In [ ]:
input_shape = X_train.shape[1:]

dropout_rate = 0.015

model_mnist_C = tf.keras.models.Sequential([
    tf.keras.layers.Input(input_shape),
    tf.keras.layers.Lambda(lambda data: data / 256.0),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(10, activation='relu'), tf.keras.layers.Dropout(rate=dropout_rate),
    tf.keras.layers.Dense(30, activation='relu'), tf.keras.layers.Dropout(rate=dropout_rate),
    tf.keras.layers.Dense(50, activation='relu'), tf.keras.layers.Dropout(rate=dropout_rate),
    tf.keras.layers.Dense(30, activation='relu'), tf.keras.layers.Dropout(rate=dropout_rate),
    tf.keras.layers.Dense(10)
])

model_mnist_C.compile(optimizer=tf.keras.optimizers.Adam(),
                      loss=tf.keras.losses.CategoricalCrossentropy(from_logits=True),
                      metrics=['accuracy'])

model_mnist_C.summary()

In [ ]:
hist_mnist_C = model_mnist_C.fit(X_train, to_categ(y_train), epochs=50, validation_split=0.2, verbose=1)

In [ ]:
plot_hist([hist_mnist_A, hist_mnist_B, hist_mnist_C], ["mnist A", "mnist B", "mnist C"])

In [ ]:
loss, acc = model_mnist_A.evaluate(X_test, to_categ(y_test))
print(f"acc={acc}, loss={loss}")

In [ ]:
loss, acc = model_mnist_B.evaluate(X_test, to_categ(y_test))
print(f"acc={acc}, loss={loss}")

In [ ]:
loss, acc = model_mnist_C.evaluate(X_test, to_categ(y_test))
print(f"acc={acc}, loss={loss}")

Мы видим, что применение dropout действительно уменьшило переобучение. Более того, качество модели также улучшилось.


## Недостатки полносвязных слоёв

В полносвязном слое каждый нейрон получает сигналы от всех нейронов предыдущего слоя.

С одной стороны, такая конфигурация охватывает все возможные варианты: если нейрону $j$ не нужен входной сигнал $i$, соответствующий вес $w_{ij}$ можно просто сделать равным нулю.

Но на практике современные методы обучения не способны работать настолько точно.

Как мы уже видели выше, полносвязные слои обладают очень большой информационной ёмкостью и поэтому склонны к переобучению — запоминанию обучающих примеров вместо извлечения общих признаков.

Другая проблема полносвязных слоёв заключается в огромном количестве обучаемых параметров.

Если полносвязный слой содержит $M$ нейронов, а предыдущий слой — $N$ нейронов, то такой слой имеет $N \times M$ весов и ещё $M$ смещений, которые необходимо обучать.

Это требует большого объёма памяти и вычислений.

Одним из решений этой проблемы является использование слоёв, в которых схема соединений адаптирована к структуре обрабатываемых данных.


## Свёртка изображения

Прежде чем переходить к слоям нейронной сети, рассмотрим широко используемый метод обработки изображений.

Он называется фильтрацией или, точнее, свёрткой (convolution).

Идея заключается в следующем.

Мы последовательно проходим по пикселям изображения.

Для каждого пикселя рассматриваются его соседние пиксели и вычисляется взвешенная сумма значений этого пикселя и его соседей.

Полученный результат становится новым значением пикселя.

Весовые коэффициенты задаются матрицей (обычно квадратной), которая называется ядром (kernel).


![convol.svg](https://raw.githubusercontent.com/kupav/data-sc-intro/refs/heads/main/fig/convol.svg)

На рисунке выше изображение размером 4 × 4 пикселя и ядро размером 3 × 3.

Показано, как ядро применяется к пикселю с координатами $y=1$, $x=1$ (нумерация начинается с нуля).

Его исходное значение равно 65. Ядро применяется следующим образом:

$$
5 \times 65 - 1\times (18 + 76 + 3 + 89) = 139
$$

Новое значение пикселя равно 139.

Иногда полученное значение может выходить за допустимые границы \[0,255\].

Например, если исходный пиксель имел бы значение 165, то новый пиксель был бы равен 639.

Существует два варианта обработки таких ситуаций.

Иногда свёртка используется для дальнейшей обработки изображения, например для обнаружения границ объектов.

В этом случае выход за диапазон не является проблемой.

Но если мы хотим получить обычное изображение, такие значения обрезаются. Значение 639 будет ограничено значением 255.

Ещё одна проблема свёртки связана с граничными пикселями.

На рисунке ниже слева показана обычная ситуация, когда ядро применяется к пикселю (1,1).

Справа ядро находится в позиции (0,0), и часть его выходит за пределы изображения.

Здесь также существует несколько вариантов решения.

Можно считать, что изображение окружено нулевыми пикселями.

Тогда при вычислении свёртки для граничных пикселей значения за пределами изображения считаются равными нулю.

Это называется нулевым дополнением (zero padding).

Другой вариант — просто не обрабатывать граничные пиксели, чтобы ядро никогда не выходило за пределы изображения.


![padding.svg](https://raw.githubusercontent.com/kupav/data-sc-intro/refs/heads/main/fig/padding.svg)

Рассмотрим, как работает свёртка на практике. Мы загрузим изображение из репозитория и обработаем его несколькими ядрами.

Прежде всего нам понадобится вспомогательная функция для загрузки изображения.


In [ ]:
# Это для загрузки рисунка из репозитория

import matplotlib.pyplot as plt
import numpy as np
import requests
from io import BytesIO
import pathlib

def load_img(file_name):
    """Downloads csv numeric dataset from repo to numpy array."""
    base_url = "https://raw.githubusercontent.com/kupav/data-sc-intro/main/data/"
    web_data = requests.get(base_url + file_name)
    assert web_data.status_code == 200

    img_type = pathlib.Path(file_name).suffix[1:]  # suffix returns extension with leading dot
    return plt.imread(BytesIO(web_data.content), img_type)

In [ ]:
image = load_img('cats.jpg')
plt.imshow(image)
plt.axis("off")
print(image.shape)

Определим функцию, выполняющую свёртку.

В обсуждении выше изображение рассматривалось как двумерный массив.

Но если изображение цветное, каждый пиксель имеет три канала: красный, зелёный и синий.

Такое изображение можно рассматривать как стек из трёх двумерных изображений.

Соответственно, ядро также может иметь третье измерение.

Или можно сказать, что ядро для цветного изображения представляет собой стек из трёх двумерных ядер.

Однако для простоты ниже используется двумерное ядро, которое применяется к каждому каналу независимо.


In [ ]:
def convol(image, kernel):
    """Convolve image with the kernel. Two dimensionl kernel is applied at
    each color channel separately. Zero padding is used."""

    # accepts only two dimensional kernels
    assert len(kernel.shape) == 2
    # accepts only square kerenels
    assert kernel.shape[0] == kernel.shape[1]
    # accepts only odd sized kernels
    k_size, _ = kernel.shape
    assert k_size % 2 == 1

    # How many zeros to add near edges
    pad = k_size // 2

    height, width, chan = image.shape
    padded_image = np.zeros((height + 2 * pad, width + 2 * pad, chan), dtype=float)
    padded_image[pad:-pad, pad:-pad] = image

    # now the image pixels are encoded by numbers betweeb 0 and 1
    padded_image /= padded_image.max()

    # the output pixels will also be between 0 and 1
    new_image = np.zeros_like(image, dtype=float)

    for y in range(pad, height + pad):
        for x in range(pad, width + pad):
            for c in range(chan):
                # extract a block of pixels
                block = padded_image[y-pad:y+pad+1, x-pad:x+pad+1, c]
                # here the kernel is applied to a chanel c
                pixel = np.sum(block * kernel)
                # clipi outliers and write new pixel
                new_image[y-pad, x-pad, c] = np.clip(pixel, 0.0, 1.0)

    return new_image

Ниже приведены три примера ядер, которые часто используются.


In [ ]:
# This kernel sharpen image
kernel_sharpen = np.array([
    [ 0, -1,  0],
    [-1,  5, -1],
    [ 0, -1,  0]])

# Detection of edges
kernel_edges = np.array([
    [-1, -1, -1],
    [-1,  8, -1],
    [-1, -1, -1]])

# Bluring
kernel_gausblur = np.array([
    [1, 4, 6, 4, 1],
    [4, 16, 24, 16, 4],
    [6, 24, 36, 24, 6],
    [4, 16, 24, 16, 4],
    [1, 4, 6, 4, 1]]) / 256.0

image_sharpen = convol(image, kernel_sharpen)
image_edges = convol(image, kernel_edges)
image_gausblur = convol(image, kernel_gausblur)

In [ ]:
fig, axs = plt.subplots(nrows=2, ncols=2, figsize=(11, 8))

axs[0, 0].imshow(image)
axs[0, 0].set_title("original")

axs[0, 1].imshow(image_sharpen)
axs[0, 1].set_title("sharpen")

axs[1, 0].imshow(image_edges)
axs[1, 0].set_title("edges")

axs[1, 1].imshow(image_gausblur)
axs[1, 1].set_title("gaussian blur")

for ax in axs.reshape(-1):
    ax.axis("off")

plt.tight_layout()

## Слой свёрточной нейронной сети

Идею обработки изображения с помощью свёртки можно использовать в нейронных сетях.

Вместо полносвязного слоя можно создать слой, выполняющий свёртку.

В отличие от обычной обработки изображений веса ядра инициализируются случайно и затем настраиваются во время обучения.

Это можно рассматривать как работу одного нейрона с ограниченным радиусом связей (используются только локальные связи), который многократно применяется к каждому элементу данных.


![conv.png](https://raw.githubusercontent.com/kupav/data-sc-intro/refs/heads/main/fig/conv.png)

Ядро в свёрточном слое является трёхмерным.

Первые два измерения задают его ширину и высоту, как обсуждалось выше, а третье измерение соответствует каналам.

Исходное цветное изображение имеет три канала: красный, зелёный и синий.

Когда изображение обрабатывается одним ядром, вычисляется новый двумерный массив пикселей, где каждый пиксель представляет собой взвешенную сумму соседних пикселей по всем каналам.


![convol_3chan.svg](https://raw.githubusercontent.com/kupav/data-sc-intro/refs/heads/main/fig/convol_3chan.svg)

Обычно свёрточный слой содержит несколько обучаемых ядер, которые применяются одновременно.

Каждое ядро создаёт новый двумерный массив, и если используется $N$ ядер, получается $N$ таких массивов.


![convol_many_chan.svg](https://raw.githubusercontent.com/kupav/data-sc-intro/refs/heads/main/fig/convol_many_chan.svg)

Каждое ядро соответствует одному нейрону.

Таким образом, при использовании нескольких ядер в свёрточном слое несколько нейронов выполняют одну и ту же задачу параллельно: все они обрабатывают весь входной образец данных.

После прохождения через свёрточный слой число каналов увеличивается.


## Pooling

Увеличение числа каналов благодаря использованию нескольких свёрточных ядер важно, поскольку множество ядер (то есть множество свёрточных нейронов) имеет больше возможностей для извлечения существенных признаков изображения.

Это своего рода соревнование: несколько свёрточных нейронов работают параллельно и выполняют одну и ту же задачу.

Но после этого необходимо каким‑то образом подвести итог и определить «победителей».

Иначе объём данных будет постоянно увеличиваться.

Для этого используются так называемые pooling‑слои.

Pooling применяется к каждому каналу отдельно, то есть к результатам каждого свёрточного нейрона без сравнения с другими.

Pooling также использует ядро. В данном случае ядро — это просто окно, которое перемещается по изображению и выполняет определённую операцию агрегирования.

Существует два типа pooling: max pooling и average pooling.

Ниже показана иллюстрация max pooling: в окне сохраняется только наибольшее значение.

Иными словами, мы берём результаты каждого свёрточного нейрона (каждый канал — это результат работы одного нейрона) и оставляем только наиболее сильные отклики (наибольшие значения в локальной области).

Мы удаляем не сами нейроны (как это происходит при dropout), а их менее значимые результаты.

При average pooling все значения внутри окна усредняются, и результат передаётся на следующий слой.

Иными словами, выполняется сглаживание каждого канала.


![pooling.svg](https://raw.githubusercontent.com/kupav/data-sc-intro/refs/heads/main/fig/pooling.svg)

Ещё один специальный случай pooling называется глобальным pooling.

Он соответствует окну размером со всё изображение.

В результате на каждом канале остаётся только одно значение — либо максимум, либо среднее.

Например, если до глобального pooling входные данные имели форму (100, 100, 32), это означает, что имеется 32 канала, и на каждом канале находится двумерный массив размером (100, 100).

Если применяется pooling с ядром 2×2, выход имеет форму (50, 50, 32).

А после глобального pooling форма становится (1, 1, 32).

Глобальный pooling обычно применяется один раз после всех свёрточных и локальных pooling‑слоёв.

По сути глобальный pooling предназначен для того, чтобы сделать сеть нечувствительной к размеру входного изображения.

Свёртка работает независимо от размера изображения, поскольку один и тот же нейрон применяется ко всем пикселям.

Полносвязные слои, напротив, имеют фиксированное число входов.

Поэтому, будучи созданными для определённого размера входных данных, они не могут работать с изображениями другого размера.

Обычно после свёрточных слоёв добавляют несколько полносвязных слоёв.

Чтобы согласовать эти две части сети, иногда используют глобальный pooling, который в итоге удаляет информацию о исходном размере изображения.


## Dropout после свёрточных слоёв

Напомним: слои dropout обычно располагаются после полносвязных (dense) слоёв и временно отключают часть нейронов во время обучения.

Это необходимо для уменьшения переобучения.

Dropout между свёрточными слоями действует иначе.

Он не может отключать свёрточные нейроны — все они продолжают обучаться даже при наличии dropout.

Вместо этого его влияние больше похоже на добавление шума в сеть.

Хотя иногда dropout успешно используется в свёрточных сетях, обычно его лучше избегать: добавление шума в сеть можно реализовать более прямыми способами.


## Типичная структура сети со свёртками

В принципе существует множество способов комбинировать описанные выше идеи.

Одна из наиболее простых схем реализации выглядит следующим образом.

Свёрточная нейронная сеть обычно включает комбинации следующих слоёв:

- свёртка с несколькими ядрами — увеличивает число каналов  
- pooling — уменьшает размер изображения  
- после нескольких таких блоков может применяться глобальный pooling  
- затем данные преобразуются (flatten), и добавляется несколько полносвязных слоёв  
- между ними может использоваться dropout для уменьшения переобучения  
- во внутренних свёрточных слоях часто применяется функция активации ReLU  
- сеть заканчивается выходным полносвязным слоем, число нейронов которого равно числу классов  
- его функция активации — softmax, а функция потерь — кросс‑энтропия


## Свёрточная сеть для MNIST


Теперь создадим сеть со свёрточными слоями, как описано выше.

Сеть будет содержать минимально необходимый набор слоёв: свёртку, затем max‑pooling, затем flatten, первый полносвязный слой, dropout и финальный полносвязный слой.


In [ ]:
input_shape = X_train.shape[1:]

dropout_rate = 0.2

model_mnist_D = tf.keras.models.Sequential([
    tf.keras.layers.Input(input_shape),
    tf.keras.layers.Lambda(lambda data: data / 256.0),
    tf.keras.layers.Reshape((*input_shape, 1)),
    tf.keras.layers.Conv2D(4, (3, 3), padding='same', activation='relu'),
    tf.keras.layers.MaxPool2D(pool_size=(2, 2)),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(20, activation='relu'),
    tf.keras.layers.Dropout(rate=dropout_rate),
    tf.keras.layers.Dense(10)
])

model_mnist_D.compile(optimizer=tf.keras.optimizers.Adam(),
                      loss=tf.keras.losses.CategoricalCrossentropy(from_logits=True),
                      metrics=['accuracy'])

model_mnist_D.summary()

In [ ]:
hist_mnist_D = model_mnist_D.fit(X_train, to_categ(y_train), epochs=50, validation_split=0.2, verbose=1)

In [ ]:
plot_hist([hist_mnist_A, hist_mnist_B, hist_mnist_C, hist_mnist_D],
          ["mnist A", "mnist B", "mnist C", "mnist D"])

In [ ]:
loss, acc = model_mnist_A.evaluate(X_test, to_categ(y_test))
print(f"acc={acc}, loss={loss}")

In [ ]:
loss, acc = model_mnist_B.evaluate(X_test, to_categ(y_test))
print(f"acc={acc}, loss={loss}")

In [ ]:
loss, acc = model_mnist_C.evaluate(X_test, to_categ(y_test))
print(f"acc={acc}, loss={loss}")

In [ ]:
loss, acc = model_mnist_D.evaluate(X_test, to_categ(y_test))
print(f"acc={acc}, loss={loss}")

Обратите внимание, что даже один свёрточный слой способен улучшить качество модели.

Обычно сети для классификации изображений содержат гораздо больше таких слоёв.
